# 🚗 Generador Automático de Proformas - AMBACAR v3
**Instrucciones:** Ejecuta las celdas en orden (▶). Para hacer otra proforma ejecuta la celda REINICIO.
---

In [ ]:
# ══════════════════════════════════════════
# 🔄 REINICIO — nueva proforma sin reiniciar
# ══════════════════════════════════════════
for v in ['uploaded_cotizacion','uploaded_plantillas','tabla_detalle','tabla_correctivos',
          'gran_total','nombre_pdf','df_plan','df_cot','total_repuestos','total_lubricantes',
          'total_mo','total_preventivo','total_rep_corr','total_lub_corr','total_mo_corr',
          'total_correctivo','total_repuestos_1v','total_lubricantes_1v','total_mo_1v',
          'total_preventivo_1v']:
    globals().pop(v, None)
print("✅ Listo. Ejecuta la Celda 1 (o salta a Celda 2 si ya instalaste las librerías).")


## 📦 CELDA 1 — Instalar librerías

In [ ]:
!pip install openpyxl pandas reportlab --quiet
print("✅ Librerías listas")


## 📂 CELDA 2 — Subir archivos
- Primero sube `DATOS_PARA_COTIZACION.xlsx`
- Luego sube la plantilla de precios

In [ ]:
from google.colab import files
import io, pandas as pd
from openpyxl import load_workbook

print("📋 Sube el archivo DATOS_PARA_COTIZACION.xlsx:")
uploaded_cotizacion = files.upload()
nombre_cotizacion = list(uploaded_cotizacion.keys())[0]
print(f"✅ Cotización: {nombre_cotizacion}")

print("\n📊 Sube la plantilla de PLAN DE MANTENIMIENTO:")
uploaded_plantillas = files.upload()
nombres_plantillas = list(uploaded_plantillas.keys())
print(f"✅ Plantilla(s): {nombres_plantillas}")


## 🔍 CELDA 3 — Leer datos de cotización

In [ ]:
df_cot = pd.read_excel(io.BytesIO(uploaded_cotizacion[nombre_cotizacion]),
                       sheet_name=0, header=None)

def buscar_valor(df, etiqueta, cols=[3,4,5,2]):
    for i, row in df.iterrows():
        c1 = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""
        if etiqueta.lower() in c1.lower():
            for c in cols:
                if c < len(row) and pd.notna(row.iloc[c]):
                    v = str(row.iloc[c]).strip()
                    if v and v.lower() not in ["nan","none",""]:
                        return v
    return None

razon_social    = buscar_valor(df_cot, "Razón Social")
ruc_cliente     = buscar_valor(df_cot, "RUC")
direccion_cli   = buscar_valor(df_cot, "Dirección")
contacto        = buscar_valor(df_cot, "Nombre del Contacto")
telefono_cli    = buscar_valor(df_cot, "Teléfono")
correo_cli      = buscar_valor(df_cot, "Correo electrónico")
plazo_ejec      = buscar_valor(df_cot, "Plazo de ejecución")
vigencia_oferta = buscar_valor(df_cot, "Vigencia de la oferta")
modelo_vehiculo = buscar_valor(df_cot, "Modelo de vehículo")
cantidad_vehic  = buscar_valor(df_cot, "Cantidad de vehículos")
observacion     = buscar_valor(df_cot, "Observación")
objeto_contrato = buscar_valor(df_cot, "OBJETO DEL CONTRATO")

# KM Desde / Hasta
km_desde = km_hasta = None
for i, row in df_cot.iterrows():
    for cc in [1,3]:
        if cc >= len(row): continue
        celda = str(row.iloc[cc]).strip() if pd.notna(row.iloc[cc]) else ""
        if celda == "Desde:":
            for vc in [cc+1,4,5]:
                if vc < len(row) and pd.notna(row.iloc[vc]):
                    try: km_desde = int(float(str(row.iloc[vc]).replace(",",""))); break
                    except: pass
        if celda == "Hasta:":
            for vc in [cc+1,4,5]:
                if vc < len(row) and pd.notna(row.iloc[vc]):
                    try: km_hasta = int(float(str(row.iloc[vc]).replace(",",""))); break
                    except: pass

if not km_desde or not km_hasta:
    for i, row in df_cot.iterrows():
        c1 = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""
        if "incluir en la cotización" in c1.lower() or "mantenimientos que se deben" in c1.lower():
            for vc in [3,4,5]:
                if vc < len(row) and pd.notna(row.iloc[vc]):
                    try:
                        v = int(float(str(row.iloc[vc]).replace(",","")))
                        if v > 1000 and not km_desde: km_desde = v; break
                    except: pass
            for j in range(i+1, min(i+5, len(df_cot))):
                r2 = df_cot.iloc[j]
                c3 = str(r2.iloc[3]).strip() if pd.notna(r2.iloc[3]) else ""
                if "Hasta" in c3 and pd.notna(r2.iloc[4]):
                    try: km_hasta = int(float(str(r2.iloc[4]).replace(",",""))); break
                    except: pass
            break

# ── Correctivos: mostrar estado detectado y pedir confirmación ──────────────
# Los checkboxes del Excel pueden desincronizarse con el estado visual.
# El programa detecta el estado y pide confirmación al usuario.
incluir_correctivos = False

import zipfile as _zf, re as _re, io as _io2

def detectar_correctivos_excel(xlsx_bytes):
    """Intenta leer el estado del checkbox de correctivos del VML del Excel."""
    try:
        with _zf.ZipFile(_io2.BytesIO(xlsx_bytes), 'r') as z:
            if 'xl/drawings/vmlDrawing1.vml' not in z.namelist():
                return None
            vml = z.read('xl/drawings/vmlDrawing1.vml').decode('utf-8', errors='ignore')
            drawing = z.read('xl/drawings/drawing1.xml').decode('utf-8', errors='ignore')
        anchors = _re.findall(
            r'<xdr:twoCellAnchor[^>]*>(.*?)</xdr:twoCellAnchor>', drawing, _re.DOTALL)
        anchor_rows = [(int(_re.search(r'<xdr:from>.*?<xdr:row>(\d+)</xdr:row>', a, _re.DOTALL).group(1))+1,
                        int(_re.search(r'<xdr:from>.*?<xdr:col>(\d+)</xdr:col>', a, _re.DOTALL).group(1))+1)
                       for a in anchors if _re.search(r'<xdr:from>.*?<xdr:row>(\d+)</xdr:row>', a, _re.DOTALL)]
        shapes_raw = _re.findall(r'<v:shape id="(?!_x0000_t)[^"]*".*?</v:shape>', vml, _re.DOTALL)
        shapes = sorted(
            [(float(_re.search(r'margin-top:([.\d]+)pt', s).group(1)) if _re.search(r'margin-top:([.\d]+)pt', s) else 0,
              '<x:Checked>' in s) for s in shapes_raw], key=lambda x: x[0])
        df_tmp = pd.read_excel(_io2.BytesIO(xlsx_bytes), sheet_name=0, header=None)
        for idx, ((mt, chk), (row, col)) in enumerate(zip(shapes, anchor_rows)):
            b = str(df_tmp.iloc[row-1, 1]).strip() if pd.notna(df_tmp.iloc[row-1, 1]) else ""
            if "correctivo" in b.lower() and "incluir" in b.lower():
                si_chk = chk
                no_chk = shapes[idx+1][1] if idx+1 < len(shapes) else False
                return (si_chk, no_chk)
    except:
        pass
    return None

xlsx_bytes = uploaded_cotizacion[nombre_cotizacion]
estado = detectar_correctivos_excel(xlsx_bytes)

if estado:
    si_chk, no_chk = estado
    detectado = si_chk and not no_chk
    print(f"\n📋 Estado detectado en el Excel:")
    print(f"   Checkbox SI : {'☑ MARCADO' if si_chk else '☐ vacío'}")
    print(f"   Checkbox NO : {'☑ MARCADO' if no_chk else '☐ vacío'}")
    print(f"\n⚠️  Los checkboxes del Excel a veces no se guardan correctamente.")
    print(f"   Por favor confirma manualmente:")
    resp = input("\n¿Se deben incluir correctivos? [S/N]: ").strip().upper()
    incluir_correctivos = resp == "S"
else:
    # Fallback texto
    for i, row in df_cot.iterrows():
        c1 = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""
        if "correctivo" in c1.lower() and "incluir" in c1.lower():
            for vc in [3,4,5]:
                if vc < len(row):
                    v = str(row.iloc[vc]).strip().upper() if pd.notna(row.iloc[vc]) else ""
                    if v == "SI": incluir_correctivos = True; break
            break
    resp = input(f"\n¿Incluir correctivos? (detectado: {'S' if incluir_correctivos else 'N'}) [S/N]: ").strip().upper()
    incluir_correctivos = resp == "S"

print(f"\n✅ Correctivos: {'SÍ incluir' if incluir_correctivos else 'NO incluir'}")

# Placas
placas = []
for i, row in df_cot.iterrows():
    c1 = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""
    if c1.upper() == "PLACAS":
        for vc in [3,4,5]:
            if vc < len(row) and pd.notna(row.iloc[vc]):
                v = str(row.iloc[vc]).strip()
                if v and v.lower() not in ["nan","placas",""]:
                    placas.append(v); break

print("=" * 55)
print("📋 DATOS DE LA COTIZACIÓN")
print("=" * 55)
print(f"  Razón social cliente : {razon_social}")
print(f"  RUC                  : {ruc_cliente}")
print(f"  Dirección            : {direccion_cli}")
print(f"  Contacto             : {contacto}")
print(f"  Teléfono             : {telefono_cli}")
print(f"  Correo               : {correo_cli}")
print(f"  Modelo de vehículo   : {modelo_vehiculo}")
print(f"  Cantidad de vehículos: {cantidad_vehic}")
print(f"  Rango KM             : {km_desde:,} – {km_hasta:,}")
print(f"  Incluir correctivos  : {'SÍ' if incluir_correctivos else 'NO'}")
print(f"  Plazo ejecución      : {plazo_ejec} días")
print(f"  Vigencia oferta      : {vigencia_oferta}")
print(f"  Observación          : {observacion}")
print(f"  Placas               : {', '.join(placas) if placas else 'No especificadas'}")
print("=" * 55)


## 🗂️ CELDA 4 — Seleccionar plantilla y hoja

In [ ]:
if len(nombres_plantillas) == 1:
    plantilla_elegida = nombres_plantillas[0]
    print(f"✅ Plantilla: {plantilla_elegida}")
else:
    for idx, p in enumerate(nombres_plantillas):
        print(f"  [{idx}] {p}")
    plantilla_elegida = nombres_plantillas[int(input("Número de plantilla: "))]

wb = load_workbook(io.BytesIO(uploaded_plantillas[plantilla_elegida]), read_only=True)
hojas = wb.sheetnames
print(f"\n📑 Hojas disponibles:")
for idx, h in enumerate(hojas): print(f"  [{idx}] {h}")

# Auto-detectar hoja según 4X4/4X2
hoja_elegida = None
modelo_upper = (modelo_vehiculo or "").upper()
excluir = ["CORRECTIVO","CONSOLID","CRONOGRAMA"]
for h in hojas:
    if any(ex in h.upper() for ex in excluir): continue
    if "4X4" in modelo_upper and "4X4" in h.upper(): hoja_elegida = h; break
    if "4X2" in modelo_upper and "4X2" in h.upper(): hoja_elegida = h; break

if hoja_elegida:
    print(f"\n🤖 Auto-detectada: '{hoja_elegida}'")
    if input("¿Usar esta hoja? [S/n]: ").strip().lower() == "n":
        hoja_elegida = None

if not hoja_elegida:
    hoja_elegida = hojas[int(input("Número de hoja: "))]

print(f"\n✅ Hoja seleccionada: '{hoja_elegida}'")

hoja_correctivos = None
if incluir_correctivos:
    for h in hojas:
        if "CORRECTIVO" in h.upper() or "REF" in h.upper():
            hoja_correctivos = h; break
    print(f"🔧 Hoja correctivos: '{hoja_correctivos}'" if hoja_correctivos else "⚠️ No se encontró hoja de correctivos.")


## 🧮 CELDA 5 — Calcular totales
**Lógica corregida:** usa solo el primer bloque de la tabla (que ya contiene todos los KMs),
filtra las columnas dentro del rango pedido y suma celda por celda.

In [ ]:
df_plan = pd.read_excel(
    io.BytesIO(uploaded_plantillas[plantilla_elegida]),
    sheet_name=hoja_elegida, header=None
)

def km_a_int(s):
    try: return int(str(s).upper().replace("KM","").replace(".","").replace(",","").strip())
    except: return None

# ── Usar SOLO el bloque 1 (primera fila con "GWM...DIESEL") ─────────────────
fila_enc = None
for i, row in df_plan.iterrows():
    c0 = str(row.iloc[0]).strip()
    if "GWM" in c0.upper() and "DIESEL" in c0.upper():
        fila_enc = i; break

if fila_enc is None:
    # Fallback: fila 1
    fila_enc = 1
    print("⚠️ No se encontró encabezado GWM, usando fila 1.")

print(f"📦 Usando bloque principal en fila {fila_enc}")

# Detectar fin del bloque 1 (siguiente bloque o final del archivo)
fin_bloque = len(df_plan)
for i in range(fila_enc + 5, len(df_plan)):
    c0 = str(df_plan.iloc[i, 0]).strip()
    if "GWM" in c0.upper() and "DIESEL" in c0.upper():
        fin_bloque = i - 1; break

# Columnas en rango
fila_km_row = df_plan.iloc[fila_enc]
cols_rango = [(ci, km_a_int(v)) for ci, v in enumerate(fila_km_row)
              if km_a_int(v) and km_desde <= km_a_int(v) <= km_hasta]

if not cols_rango:
    print("❌ No se encontraron columnas en el rango especificado.")
    print(f"   KMs disponibles en esta hoja:")
    kms_disp = [km_a_int(v) for v in fila_km_row if km_a_int(v)]
    print(f"   {sorted(kms_disp)}")
else:
    print(f"✅ {len(cols_rango)} mantenimientos en rango [{km_desde:,} – {km_hasta:,} KM]")
    print(f"   Desde {cols_rango[0][1]:,} KM hasta {cols_rango[-1][1]:,} KM")

# Buscar filas de totales dentro del bloque 1
fila_rep = fila_lub = fila_mo = fila_tot = None
for fi in range(fila_enc, fin_bloque):
    etiq = str(df_plan.iloc[fi, 0]).strip().upper()
    if etiq == "TOTAL REPUESTOS"          and fila_rep is None: fila_rep = fi
    if etiq == "TOTAL LUBRICANTES"        and fila_lub is None: fila_lub = fi
    if etiq == "TOTAL M/O"               and fila_mo  is None: fila_mo  = fi
    if etiq == "TOTAL MANTENIMIENTO X KM" and fila_tot is None: fila_tot = fi

print(f"📍 Filas de totales: REPUESTOS={fila_rep}, LUBRICANTES={fila_lub}, M/O={fila_mo}")

def gv(fi, ci):
    try: return float(df_plan.iloc[fi, ci]) if pd.notna(df_plan.iloc[fi, ci]) else 0.0
    except: return 0.0

# Sumar solo las celdas del rango para 1 vehículo
total_repuestos_1v   = sum(gv(fila_rep, ci) for ci,_ in cols_rango)
total_lubricantes_1v = sum(gv(fila_lub, ci) for ci,_ in cols_rango)
total_mo_1v          = sum(gv(fila_mo,  ci) for ci,_ in cols_rango)
total_preventivo_1v  = total_repuestos_1v + total_lubricantes_1v + total_mo_1v

# Detalle por KM para la tabla del PDF
tabla_detalle = []
for ci, km in cols_rango:
    rep = gv(fila_rep, ci); lub = gv(fila_lub, ci); mo = gv(fila_mo, ci)
    tot = gv(fila_tot, ci) if fila_tot else rep+lub+mo
    tabla_detalle.append({"km":km,"repuestos":rep,"lubricantes":lub,"mano_obra":mo,"total":tot})

# Multiplicar por cantidad de vehículos
n_vehiculos       = int(float(str(cantidad_vehic).replace(",",".")))
total_repuestos   = total_repuestos_1v   * n_vehiculos
total_lubricantes = total_lubricantes_1v * n_vehiculos
total_mo          = total_mo_1v          * n_vehiculos
total_preventivo  = total_preventivo_1v  * n_vehiculos

print(f"\n{'='*55}")
print(f"💰 RESUMEN PREVENTIVO — {n_vehiculos} vehículos")
print(f"{'='*55}")
print(f"  Total Repuestos    : $ {total_repuestos:>12,.2f}")
print(f"  Total Lubricantes  : $ {total_lubricantes:>12,.2f}")
print(f"  Total Mano de Obra : $ {total_mo:>12,.2f}")
print(f"  {'─'*38}")
print(f"  TOTAL PREVENTIVO   : $ {total_preventivo:>12,.2f}")
print(f"  Mantenimientos     : {len(tabla_detalle)}")
print(f"{'='*55}")


## 🔧 CELDA 6 — Correctivos (si aplica)

In [ ]:
total_rep_corr = total_lub_corr = total_mo_corr = total_correctivo = 0.0
tabla_correctivos = []

if incluir_correctivos:
    # ── Siempre usar 30% del total preventivo ───────────────────────────────
    # (La hoja REF MANT CORRECTIVO contiene precios referenciales unitarios,
    #  no el total real del contrato. El consolidado oficial usa el 30%.)
    total_correctivo = round(total_preventivo * 0.30, 2)
    print(f"✅ Correctivo = 30% del total preventivo")
    print(f"   30% de $ {total_preventivo:,.2f} = $ {total_correctivo:,.2f}")
else:
    total_correctivo = 0.0
    print("ℹ️  Correctivos NO incluidos en esta cotización.")

gran_total = total_preventivo + total_correctivo
print(f"\n{'='*55}")
print(f"🔧 CORRECTIVO     : $ {total_correctivo:>12,.2f}")
print(f"💰 PREVENTIVO     : $ {total_preventivo:>12,.2f}")
print(f"  {'─'*38}")
print(f"💵 GRAN TOTAL     : $ {gran_total:>12,.2f}")
print(f"{'='*55}")


## 📄 CELDA 7 — Generar PDF

In [ ]:
!pip install python-docx Pillow --quiet
from docx import Document
from docx.shared import Pt, Cm, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_ALIGN_VERTICAL
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from datetime import date
import requests, base64, io as _io, os as _os

today = date.today()
num_proforma = f"AMB-PV-{today.year}-AUTO-{today.strftime('%m%d')}"
meses = {"January":"enero","February":"febrero","March":"marzo","April":"abril",
         "May":"mayo","June":"junio","July":"julio","August":"agosto",
         "September":"septiembre","October":"octubre","November":"noviembre","December":"diciembre"}
fecha_str = today.strftime("%d de %B de %Y")
for en, es in meses.items(): fecha_str = fecha_str.replace(en, es)

nombre_docx = f"Proforma_{num_proforma}.docx"

# ── Subir imágenes (solo primera vez) ────────────────────────────────────────
from google.colab import files as _gfiles
import shutil as _sh

header_path = "header_ambacar.png"
footer_path  = "footer_ambacar.png"

if not _os.path.exists(header_path):
    print("📎 Sube la imagen del HEADER (logo ambacar+HAVAL+GW):")
    up = _gfiles.upload()
    _sh.copy(list(up.keys())[0], header_path)

if not _os.path.exists(footer_path):
    print("📎 Sube la imagen del FOOTER (línea roja + datos Ambacar):")
    up = _gfiles.upload()
    _sh.copy(list(up.keys())[0], footer_path)

# ── Helpers ──────────────────────────────────────────────────────────────────
ROJO = RGBColor(0xCC, 0x00, 0x00)

def set_cell_bg(cell, hex_color):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'), hex_color)
    tcPr.append(shd)

def set_cell_border(cell, **kwargs):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcBorders = OxmlElement('w:tcBorders')
    for side in ['top','left','bottom','right']:
        border = OxmlElement(f'w:{side}')
        border.set(qn('w:val'), kwargs.get('val','single'))
        border.set(qn('w:sz'), kwargs.get('sz','4'))
        border.set(qn('w:space'), '0')
        border.set(qn('w:color'), kwargs.get('color','999999'))
        tcBorders.append(border)
    tcPr.append(tcBorders)

def p_run(para, text, bold=False, italic=False, size=9, color=None, underline=False):
    run = para.add_run(text)
    run.bold = bold; run.italic = italic; run.underline = underline
    run.font.size = Pt(size)
    if color: run.font.color.rgb = color
    return run

def add_para(doc, text="", bold=False, italic=False, size=9, align=WD_ALIGN_PARAGRAPH.JUSTIFY,
             space_before=0, space_after=4, left_indent=0, color=None):
    p = doc.add_paragraph()
    p.alignment = align
    p.paragraph_format.space_before = Pt(space_before)
    p.paragraph_format.space_after  = Pt(space_after)
    if left_indent: p.paragraph_format.left_indent = Cm(left_indent)
    if text:
        r = p.add_run(text); r.bold=bold; r.italic=italic; r.font.size=Pt(size)
        if color: r.font.color.rgb = color
    return p

def add_heading(doc, text, size=10, color=ROJO, space_before=6, space_after=3):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(space_before)
    p.paragraph_format.space_after  = Pt(space_after)
    r = p.add_run(text); r.bold=True; r.font.size=Pt(size)
    if color: r.font.color.rgb = color
    return p

def add_bullet(doc, text, level=0, size=9, bold_prefix=None):
    p = doc.add_paragraph(style='List Bullet' if level==0 else 'List Bullet 2')
    p.paragraph_format.space_after = Pt(2)
    p.paragraph_format.left_indent = Cm(0.5 + level*0.5)
    if bold_prefix:
        r = p.add_run(bold_prefix); r.bold=True; r.font.size=Pt(size)
    r = p.add_run(text); r.font.size=Pt(size)
    return p

# ── Documento ─────────────────────────────────────────────────────────────────
doc = Document()

# Márgenes A4
for section in doc.sections:
    section.page_width  = Cm(21)
    section.page_height = Cm(29.7)
    section.left_margin = section.right_margin = Cm(2)
    section.top_margin    = Cm(2)
    section.bottom_margin = Cm(2)

# ── HEADER con imagen ─────────────────────────────────────────────────────────
section = doc.sections[0]
header = section.header
header.is_linked_to_previous = False
hp = header.paragraphs[0] if header.paragraphs else header.add_paragraph()
hp.alignment = WD_ALIGN_PARAGRAPH.LEFT
hp.paragraph_format.space_after = Pt(0)
# Agregar imagen header
run_h = hp.add_run()
run_h.add_picture(header_path, width=Cm(17))

# ── FOOTER con imagen ─────────────────────────────────────────────────────────
footer = section.footer
footer.is_linked_to_previous = False
fp = footer.paragraphs[0] if footer.paragraphs else footer.add_paragraph()
fp.alignment = WD_ALIGN_PARAGRAPH.LEFT
fp.paragraph_format.space_before = Pt(0)
run_f = fp.add_run()
run_f.add_picture(footer_path, width=Cm(17))

# ── FECHA y NÚMERO DE PROFORMA ────────────────────────────────────────────────
p_fecha = doc.add_paragraph()
p_fecha.alignment = WD_ALIGN_PARAGRAPH.RIGHT
p_fecha.paragraph_format.space_after = Pt(0)
p_run(p_fecha, f"Quito, {fecha_str}", size=9)

p_num = doc.add_paragraph()
p_num.alignment = WD_ALIGN_PARAGRAPH.RIGHT
p_num.paragraph_format.space_after = Pt(6)
p_run(p_num, f"Proforma Nro. {num_proforma}", size=9)

# Línea separadora roja
p_line = doc.add_paragraph()
p_line.paragraph_format.space_after = Pt(6)
pPr = p_line._p.get_or_add_pPr()
pBdr = OxmlElement('w:pBdr')
bottom = OxmlElement('w:bottom')
bottom.set(qn('w:val'), 'single'); bottom.set(qn('w:sz'), '6')
bottom.set(qn('w:space'), '1'); bottom.set(qn('w:color'), 'CC0000')
pBdr.append(bottom); pPr.append(pBdr)

# ── TÍTULO ────────────────────────────────────────────────────────────────────
p_tit = doc.add_paragraph()
p_tit.alignment = WD_ALIGN_PARAGRAPH.CENTER
p_tit.paragraph_format.space_after = Pt(8)
r_tit = p_tit.add_run(objeto_contrato or "Plan de mantenimiento preventivo y correctivo de vehículos")
r_tit.bold = True; r_tit.font.size = Pt(13); r_tit.font.color.rgb = ROJO

# ── DATOS DEL PROVEEDOR ──────────────────────────────────────────────────────
add_heading(doc, "DATOS DEL PROVEEDOR:")
for lbl, val in [
    ("Razón Social: ", "AMBACAR CIA LTDA."),
    ("RUC: ", "1890010705001"),
    ("Dirección: ", "Av. Indoamérica Km1, Ambato."),
    ("Teléfonos de contacto: ", "0983509888 - 0993758313"),
    ("Persona de Contacto: ", "Geovanna Pilapanta / Christian Salazar / Byron López"),
    ("Correo Electrónico: ", "admincontratacion@ambacar.com / contratacionpublica02@ambacar.com / contratacionpublica04@ambacar.com"),
]:
    p = doc.add_paragraph()
    p.paragraph_format.left_indent = Cm(0.3)
    p.paragraph_format.space_after = Pt(2)
    p_run(p, "- ", size=9)
    p_run(p, lbl, bold=True, size=9)
    p_run(p, val, size=9)

doc.add_paragraph().paragraph_format.space_after = Pt(2)

# ── DATOS DEL CLIENTE (tabla) ─────────────────────────────────────────────────
add_heading(doc, "DATOS DEL CLIENTE:")
tbl_cli = doc.add_table(rows=5, cols=2)
tbl_cli.alignment = WD_TABLE_ALIGNMENT.LEFT
tbl_cli.style = 'Table Grid'
col_widths = [Cm(4), Cm(13)]
datos_cli = [
    ("Razón Social:", razon_social or ""),
    ("RUC:", ruc_cliente or ""),
    ("Dirección:", direccion_cli or ""),
    ("Teléfono:", str(telefono_cli) if telefono_cli else ""),
    ("Correo electrónico:", correo_cli or ""),
]
for i, (lbl, val) in enumerate(datos_cli):
    row = tbl_cli.rows[i]
    row.cells[0].width = col_widths[0]; row.cells[1].width = col_widths[1]
    row.cells[0].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
    row.cells[1].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
    set_cell_bg(row.cells[0], "E0E0E0")
    p0 = row.cells[0].paragraphs[0]
    p0.paragraph_format.space_after = Pt(0)
    p_run(p0, lbl, bold=True, size=9)
    p1 = row.cells[1].paragraphs[0]
    p1.paragraph_format.space_after = Pt(0)
    p_run(p1, val, size=9)

doc.add_paragraph().paragraph_format.space_after = Pt(4)

# ── COMPONENTES OFERTADOS ─────────────────────────────────────────────────────
add_heading(doc, "COMPONENTES OFERTADOS:")
tbl_cpc = doc.add_table(rows=2, cols=2)
tbl_cpc.alignment = WD_TABLE_ALIGNMENT.LEFT
tbl_cpc.style = 'Table Grid'
for i, (lbl, val) in enumerate([("CODIGO CPC", "DESCRIPCIÓN"),
                                  ("87141", "SERVICIOS DE MANTENIMIENTO Y REPARACION DE VEHÍCULOS DE MOTOR.")]):
    row = tbl_cpc.rows[i]
    row.cells[0].width = Cm(3); row.cells[1].width = Cm(14)
    for c in [0,1]:
        row.cells[c].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
        if i == 0: set_cell_bg(row.cells[c], "E0E0E0")
        p = row.cells[c].paragraphs[0]
        p.paragraph_format.space_after = Pt(0)
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER if c==0 else WD_ALIGN_PARAGRAPH.LEFT
        p_run(p, lbl if c==0 else val, bold=(i==0), size=9)

doc.add_paragraph().paragraph_format.space_after = Pt(4)

# ── MANTENIMIENTO PREVENTIVO — texto legal ────────────────────────────────────
add_heading(doc, "MANTENIMIENTO PREVENTIVO")
add_para(doc, "El mencionado mantenimiento conlleva la programación de inspecciones, tanto de funcionamiento "
    "como de seguridad, ajustes, reparaciones o cambio de repuestos, análisis, limpieza, cambio de "
    "lubricantes, calibración, mano de obra, entre otras, que deben desarrollarse de forma periódica con "
    "base en la planificación establecida por el proveedor autorizado AMBACAR CIA. LTDA., conforme "
    "se determina a continuación:", size=9)
add_para(doc, "Considerando la información remitida por el proveedor autorizado para brindar el servicio, este tipo "
    "de mantenimiento debe llevarse a cabo considerándose los plazos establecidos para cada vehículo "
    "(tiempo, kilometro, recorrido), esto es cada 5.000 Km, sin dejar de mencionar que los mismos se "
    "realizan en condiciones normales, conforme el siguiente detalle:", size=9)

for letra, texto in [
    ("a) ", "El proveedor deberá garantizar el cumplimiento de la garantía técnica en cumplimiento del principio de vigencia tecnológica establecido en la normativa legal vigente, en todos los trabajos desarrollados por el contratista."),
    ("b) ", "El proveedor realizará este mantenimiento en coordinación con el Administrador/a del contrato y tendrá lugar antes de que ocurran las fallas o averías en los vehículos."),
    ("c) ", "El oferente deberá disponer del recurso humano técnico y calificado, maquinaria, equipo y herramientas suficientes y necesarias para la prestación del servicio contratado, en sujeción a estos términos de referencia."),
    ("d) ", "El proveedor designará un funcionario quien conjuntamente con el Administrador/a del contrato coordinará de manera eficiente y eficaz todas las actividades relacionadas con la prestación de servicio, tanto para la solicitud de atención de vehículos, así como la tramitación de las facturas correspondientes."),
    ("e) ", "Los trabajos de mantenimiento preventivo de los vehículos serán efectuados de acuerdo al Plan de Mantenimiento Preventivo y Correctivo otorgado por el proveedor autorizado: AMBACAR Cía. Ltda."),
]:
    p = doc.add_paragraph()
    p.paragraph_format.left_indent = Cm(0.5)
    p.paragraph_format.space_after = Pt(2)
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    p_run(p, letra, bold=True, size=9)
    p_run(p, texto, size=9)

doc.add_paragraph().paragraph_format.space_after = Pt(2)

# ── MANTENIMIENTO CORRECTIVO — texto legal ────────────────────────────────────
add_heading(doc, "MANTENIMIENTO CORRECTIVO")
for txt in [
    "El mantenimiento correctivo es aplicado en casos específicos cuando los vehículos sufran desperfectos mecánicos inesperados, debido a imprevistos y/o operación anormal del vehículo por parte del/los conductores y/o usuarios; los cuales no se encuentren cubiertos por la garantía técnica otorgada por el fabricante de las camionetas.",
    "Para la realización de trabajos de mantenimiento correctivos; y establecer el presupuesto de este rubro, se deberá utilizar el catálogo referencial de valores de mantenimiento correctivo, para el modelo de camionetas adquiridas, otorgado por el proveedor del servicio AMBACAR CIA. LTDA.",
    "El mantenimiento correctivo es estimado, ya que su concurrencia no puede ser planificada. Los costos de repuestos y trabajos que se ejecutaren pueden variar de acuerdo con daños ocultos, los mismos que puede aumentar o disminuir los rubros relacionados a repuestos y mano de obra.",
    "En los casos que exista un desperfecto no contemplado para los trabajos de mantenimiento correctivo, el taller deberá contar con todos los equipos necesarios a fin de atender el requerimiento de reparación de manera ágil y oportuna evitando de esta manera que el vehículo deba permanecer inoperativo por un periodo de tiempo prolongado, excepto en los casos que la rehabilitación amerite.",
]:
    add_para(doc, txt, size=9)

add_para(doc, "Adicionalmente, se deberá considerar los siguientes aspectos:", size=9)

for letra, texto in [
    ("a) ", "Entiéndase como trabajos correctivos aquellos que por ser de carácter imprevisible técnicamente no pueden contemplarse en el Plan de Mantenimiento Preventivo y/o garantía técnica otorgada por el fabricante."),
    ("b) ", "El proveedor tendrá la responsabilidad de realizar la corrección y reparación de averías o fallas mecánicas o de cualquier índole producida de manera espontánea en los vehículos, en base a la orden de mantenimiento."),
    ("c) ", "El proveedor deberá contar con un historial de mantenimientos de la flota vehicular."),
    ("d) ", "El proveedor deberá garantizar el funcionamiento de los automotores una vez salidos del taller, en caso de presentarse inconvenientes con los trabajos mecánicos realizados, se reingresará el vehículo y se solicitará una nueva revisión sin que esta genere costos adicionales."),
    ("e) ", "El proveedor deberá garantizar el stock suficiente, los materiales, repuestos, aditivos o accesorios automotrices, los mismos deben de cumplir con las especificaciones establecidas por el fabricante, con la finalidad de asegurar el cumplimiento de la garantía técnica y el principio de vigencia tecnológica."),
    ("f) ", "El proveedor deberá garantizar el cumplimiento de la garantía técnica en cumplimiento del principio de vigencia tecnológica establecido en la normativa legal vigente, en todos los trabajos desarrollados por el contratista."),
]:
    p = doc.add_paragraph()
    p.paragraph_format.left_indent = Cm(0.5)
    p.paragraph_format.space_after = Pt(2)
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    p_run(p, letra, bold=True, size=9)
    p_run(p, texto, size=9)

doc.add_paragraph().paragraph_format.space_after = Pt(2)

# Párrafos en cursiva (NOTA)
for bold_prefix, txt, is_italic in [
    ("NOTA: ", "El detalle de mantenimientos antes referido ha sido realizado con base a la información proporcionada por el fabricante, y podrá ser modificado considerando las condiciones operativas y necesidades de la flota vehicular.", True),
    ("", "Es decir, en el caso de que en el mantenimiento correctivo se consideren varios componentes o ítems que no se encuentren establecidos en los términos de referencia y/o contrato, el contratista deberá emitir un informe técnico y proforma en donde se evidencie que el mantenimiento es conveniente para los intereses institucionales y a su vez mantener la operatividad del vehículo, sin perder la garantía técnica en cumplimiento del principio de vigencia tecnológica.", True),
    ("", "Por necesidad institucional, los trabajos o el número de veces de ejecución de los ítems indicados podrían variar. Es importante indicar que dichos ítems podrán ser aplicados a una o a todas las camionetas adquiridas a través del Catálogo Electrónico.", True),
    ("", "Los costos de repuestos y lubricantes son referenciales, y pueden estar sujetos a cambios, debido condiciones de mercado y/o a medidas económicas que afecten directamente a su distribución.", True),
]:
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    p.paragraph_format.space_after = Pt(3)
    if bold_prefix:
        p_run(p, bold_prefix, bold=True, italic=True, size=9)
    p_run(p, txt, italic=True, size=9)

doc.add_paragraph().paragraph_format.space_after = Pt(4)

# ── CONSOLIDADO DE VALORES ────────────────────────────────────────────────────
add_heading(doc, "CONSOLIDADO DE VALORES")

desc_correctivo = ("Se considera un valor aproximado del 30% del costo total de mantenimientos preventivos"
                   if incluir_correctivos and not tabla_correctivos
                   else "Mantenimiento correctivo (ítems detallados)" if tabla_correctivos else "")

n_filas_cons = 6 if incluir_correctivos else 5
tbl_cons = doc.add_table(rows=n_filas_cons, cols=4)
tbl_cons.alignment = WD_TABLE_ALIGNMENT.LEFT
tbl_cons.style = 'Table Grid'
cws = [Cm(1), Cm(4), Cm(8), Cm(4)]

# Fila 0: cabecera roja
row0 = tbl_cons.rows[0]
for c in range(4):
    row0.cells[c].width = cws[c]
    set_cell_bg(row0.cells[c], "CC0000")
# Merge toda la fila 0
row0.cells[0].merge(row0.cells[3])
p0 = row0.cells[0].paragraphs[0]
p0.alignment = WD_ALIGN_PARAGRAPH.CENTER
p0.paragraph_format.space_after = Pt(0)
p_run(p0, f"COSTOS MANTENIMIENTO {hoja_elegida.upper()}", bold=True, size=9, color=RGBColor(0xFF,0xFF,0xFF))
# Segunda línea en la misma celda
p0b = row0.cells[0].add_paragraph(f"PARA {n_vehiculos} VEHÍCULOS")
p0b.alignment = WD_ALIGN_PARAGRAPH.CENTER
p0b.paragraph_format.space_after = Pt(0)
p0b.runs[0].bold = True; p0b.runs[0].font.size = Pt(9)
p0b.runs[0].font.color.rgb = RGBColor(0xFF,0xFF,0xFF)

# Filas 1-3: preventivo
datos_prev = [
    ("1", "Total Repuestos", "Mantenimiento preventivo", f"$ {total_repuestos:,.2f}"),
    ("2", "Total Lubricantes", "", f"$ {total_lubricantes:,.2f}"),
    ("3", "Total Mano de Obra", "", f"$ {total_mo:,.2f}"),
]
for ri, (num, lbl, tipo, val) in enumerate(datos_prev, start=1):
    row = tbl_cons.rows[ri]
    for ci, (txt, bold, align, width) in enumerate([
        (num, False, WD_ALIGN_PARAGRAPH.CENTER, cws[0]),
        (lbl, False, WD_ALIGN_PARAGRAPH.LEFT, cws[1]),
        (tipo, False, WD_ALIGN_PARAGRAPH.LEFT, cws[2]),
        (val, False, WD_ALIGN_PARAGRAPH.RIGHT, cws[3]),
    ]):
        row.cells[ci].width = width
        row.cells[ci].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
        p = row.cells[ci].paragraphs[0]
        p.alignment = align; p.paragraph_format.space_after = Pt(0)
        p_run(p, txt, bold=bold, size=9)
        if ri in [2,3] and ci == 2:
            set_cell_bg(row.cells[ci], "F5F5F5")

# Merge celda "Mantenimiento preventivo" en filas 1-3 col 2
tbl_cons.rows[1].cells[2].merge(tbl_cons.rows[3].cells[2])

# Fila 4: correctivo
if incluir_correctivos:
    row4 = tbl_cons.rows[4]
    for ci, (txt, bold, align, width) in enumerate([
        ("4", False, WD_ALIGN_PARAGRAPH.CENTER, cws[0]),
        ("Mantenimiento correctivo", False, WD_ALIGN_PARAGRAPH.LEFT, cws[1]),
        (desc_correctivo, False, WD_ALIGN_PARAGRAPH.LEFT, cws[2]),
        (f"$ {total_correctivo:,.2f}", False, WD_ALIGN_PARAGRAPH.RIGHT, cws[3]),
    ]):
        row4.cells[ci].width = width
        row4.cells[ci].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
        p = row4.cells[ci].paragraphs[0]
        p.alignment = align; p.paragraph_format.space_after = Pt(0)
        p_run(p, txt, size=9)
    fila_total = 5
else:
    fila_total = 4

# Fila total
row_tot = tbl_cons.rows[fila_total]
lbl_total = "Total 1+2+3+4" if incluir_correctivos else "Total 1+2+3"
for ci, (txt, bold, align, width, bg) in enumerate([
    ("", False, WD_ALIGN_PARAGRAPH.CENTER, cws[0], None),
    ("", False, WD_ALIGN_PARAGRAPH.LEFT, cws[1], None),
    (lbl_total, True, WD_ALIGN_PARAGRAPH.RIGHT, cws[2], "FFCCCC"),
    (f"$ {gran_total:,.2f}", True, WD_ALIGN_PARAGRAPH.RIGHT, cws[3], "FFCCCC"),
]):
    row_tot.cells[ci].width = width
    if bg: set_cell_bg(row_tot.cells[ci], bg)
    p = row_tot.cells[ci].paragraphs[0]
    p.alignment = align; p.paragraph_format.space_after = Pt(0)
    p_run(p, txt, bold=bold, size=9)

p_iva = doc.add_paragraph()
p_iva.alignment = WD_ALIGN_PARAGRAPH.CENTER
p_iva.paragraph_format.space_after = Pt(6)
p_run(p_iva, "Costos no incluyen IVA", italic=True, size=8)

# ── OBSERVACIONES ─────────────────────────────────────────────────────────────
add_heading(doc, "OBSERVACIONES:")

def obs_viñeta(doc, text, bold_prefix=None, level=0):
    p = doc.add_paragraph()
    p.paragraph_format.left_indent  = Cm(0.5 + level*0.5)
    p.paragraph_format.first_line_indent = Cm(-0.3)
    p.paragraph_format.space_after  = Pt(3)
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    prefix = "- " if level == 0 else "o  "
    p_run(p, prefix, size=9)
    if bold_prefix:
        p_run(p, bold_prefix, bold=True, size=9)
    p_run(p, text, size=9)
    return p

obs_viñeta(doc, f"Plazo de ejecución de los servicios: {plazo_ejec or 365} días calendario, contados a partir del día siguiente de la fecha de suscripción de la orden de compra o contrato.")
obs_viñeta(doc, "Lugar de ejecución del servicio: en las instalaciones de AMBACAR CIA LTDA, a nivel nacional.")
obs_viñeta(doc, "Forma de pago:")
obs_viñeta(doc, "100% contra prestación de los servicios parcial, objeto del contrato, de manera mensual.", level=1)
obs_viñeta(doc, f"Mantenimientos preventivos y/o correctivos: se cancelarán una vez que los servicios sean recibidos a satisfacción por parte de la {razon_social or 'ENTIDAD CONTRATANTE'}, se haya rendido la garantía técnica y suscrito el acta de entrega recepción parcial y/o definitiva según corresponda.", level=1)

p_gtec = doc.add_paragraph()
p_gtec.paragraph_format.left_indent = Cm(0.5)
p_gtec.paragraph_format.first_line_indent = Cm(-0.3)
p_gtec.paragraph_format.space_after = Pt(0)
p_gtec.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
p_run(p_gtec, "- ", size=9)
p_run(p_gtec, "Garantía Técnica: ", bold=True, size=9)
p_run(p_gtec, "Se otorgará una garantía técnica que cubra la calidad de los servicios de mantenimiento preventivo y correctivo, así como de los repuestos utilizados. Dicha garantía tendrá una vigencia de un (1) año para los repuestos, cubriendo defectos de fabricación, y de seis (6) meses para la mano de obra correspondiente a los trabajos ejecutados.", size=9)

p_gtec2 = doc.add_paragraph()
p_gtec2.paragraph_format.left_indent  = Cm(0.8)
p_gtec2.paragraph_format.space_after  = Pt(3)
p_gtec2.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
p_run(p_gtec2, "La garantía deberá ser presentada al momento de la suscripción de la Orden de Compra o del Contrato, y permanecerá vigente hasta la finalización del último mantenimiento realizado.", size=9)

obs_viñeta(doc, f"Vigencia de la Proforma: {vigencia_oferta or '90 días'} calendario, contados a partir de su emisión.")

# ── FIRMA ─────────────────────────────────────────────────────────────────────
doc.add_paragraph().paragraph_format.space_after = Pt(4)
add_para(doc, "Atentamente,", size=9, space_after=30)
add_para(doc, "Ing. Luis Vintimilla", size=9, space_after=0)
add_para(doc, "APODERADO ESPECIAL", bold=True, size=9, space_after=0)
add_para(doc, "AMBACAR CIA. LTDA.", bold=True, size=9, space_after=0)

# ── Guardar ───────────────────────────────────────────────────────────────────
doc.save(nombre_docx)
print(f"\n✅ Word generado: {nombre_docx}")


## 💾 CELDA 8 — Descargar PDF

In [ ]:
from google.colab import files
files.download(nombre_docx)
print(f"📥 Descargando: {nombre_docx}")
